In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('/content/global_energy_mix.csv')

source_category = {
    'Coal': 'Fossil',
    'Oil': 'Fossil',
    'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon',
    'Hydro': 'Low-carbon',
    'Wind': 'Renewable',
    'Solar': 'Renewable',
    'Other Renewables': 'Renewable'
}

df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))

Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


In [2]:

# Task 1: Fossil fuel dependency by country

# Filter fossil fuels only
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# CVD-safe colour palette (Okabe-Ito inspired)
fossil_colors = {
    'Coal': '#0072B2',
    'Oil': '#E69F00',
    'Natural Gas': '#009E73'
}

# Find country with the highest fossil fuel consumption
country_totals = (
    df_fossil.groupby('Country')['TWh']
    .sum()
    .sort_values(ascending=False)
)

top_country = country_totals.index[0]

# Find region with the highest fossil fuel consumption
region_totals = df_fossil.groupby('Region')['TWh'].sum()
top_region = region_totals.idxmax()

# Create treemap
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    title=f'{top_country} leads fossil fuel consumption in {top_region}'
)

fig.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    marker=dict(line=dict(width=1, color='white'))
)

fig.update_layout(
    title_x=0.5,
    margin=dict(t=80, l=10, r=10, b=10)
)

fig.show()

print("Top fossil fuel consuming country:", top_country)
print("Top fossil fuel consuming region:", top_region)
print("Fossil fuel records:", len(df_fossil))


Top fossil fuel consuming country: Saudi Arabia
Top fossil fuel consuming region: Asia
Fossil fuel records: 43


In [3]:

# Task 2: Low-carbon energy by country

# Filter low-carbon sources
df_low = df.loc[
    df['Source_Type'] == 'Low-carbon'
].copy()

# Aggregate TWh by country
low_country = (
    df_low.groupby('Country', as_index=False)['TWh']
    .sum()
    .sort_values('TWh', ascending=False)
)

# Add dummy root node
low_country['All'] = 'Low-carbon'

# Leading country
leader = low_country.iloc[0]['Country']
leader_twh = low_country.iloc[0]['TWh']

# Treemap
fig1 = px.treemap(
    low_country,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-carbon energy distribution across countries'
)

fig1.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh'
)

fig1.update_layout(title_x=0.5)
fig1.show()

# Horizontal bar chart
fig2 = px.bar(
    low_country.sort_values('TWh', ascending=True),
    x='TWh',
    y='Country',
    orientation='h',
    color_discrete_sequence=['#0072B2'],
    title=f'{leader} leads low-carbon energy production with {leader_twh:,.0f} TWh',
    labels={'TWh': 'Low-carbon energy (TWh)', 'Country': ''}
)

fig2.update_layout(
    title_x=0.5,
    showlegend=False,
    xaxis_title='Low-carbon energy (TWh)'
)

fig2.show()

print("Leading country:", leader)
print("Low-carbon energy:", round(leader_twh, 2), "TWh")


Leading country: France
Low-carbon energy: 7593.0 TWh


The horizontal bar chart tells the ranking story more clearly because countries are arranged from lowest to highest low-carbon energy production. It is easier to compare values and identify the leading country. The treemap is useful for showing the part-to-whole relationship and the relative contribution of each country, but comparing exact rankings is more difficult.

In [4]:

# Task 3: Which region relies most on coal?

# Aggregate fossil TWh by Region and Source
region_source = (
    df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh']
    .sum()
)

# Calculate percentage share within each region
region_source['Region_Total'] = (
    region_source.groupby('Region')['TWh'].transform('sum')
)

region_source['Share_pct'] = (
    region_source['TWh'] /
    region_source['Region_Total'] * 100
)

# Find coal share for each region
coal_shares = (
    region_source[region_source['Source'] == 'Coal']
    .sort_values('Share_pct', ascending=False)
)

top_coal_region = coal_shares.iloc[0]['Region']
top_coal_share = coal_shares.iloc[0]['Share_pct']

# Coal highlighted, other sources grey
task3_colors = {
    'Coal': '#0072B2',
    'Oil': '#BDBDBD',
    'Natural Gas': '#737373'
}

# Add percentage labels only to Coal
region_source['Coal_Label'] = np.where(
    region_source['Source'] == 'Coal',
    region_source['Share_pct'].map(lambda x: f'{x:.1f}%'),
    ''
)

# Sort regions by coal share, highest first
region_order = coal_shares['Region'].tolist()

# Create horizontal stacked bar chart
fig = px.bar(
    region_source,
    x='Share_pct',
    y='Region',
    color='Source',
    text='Coal_Label',
    orientation='h',
    barmode='stack',
    category_orders={
        'Region': region_order[::-1],
        'Source': ['Coal', 'Oil', 'Natural Gas']
    },
    color_discrete_map=task3_colors,
    title=f'{top_coal_region} has the highest coal dependency ({top_coal_share:.1f}%)',
    labels={
        'Share_pct': 'Share of fossil energy (%)',
        'Region': '',
        'Source': 'Energy source'
    }
)

fig.update_traces(
    textposition='inside',
    insidetextanchor='middle'
)

fig.update_layout(
    xaxis=dict(range=[0, 100], ticksuffix='%'),
    title_x=0.5,
    legend_title='Fossil fuel source'
)

fig.show()

print("Most coal-dependent region:", top_coal_region)
print("Coal share:", round(top_coal_share, 2), "%")


Most coal-dependent region: Africa
Coal share: 78.04 %


The treemap answers which countries and energy sources contribute most to total fossil fuel consumption. It shows the hierarchical relationship between regions, countries and energy sources.

The 100% stacked bar chart answers which region has the highest coal dependency compared with Oil and Natural Gas. It makes it easier to compare the percentage composition of fossil fuels across regions.